In [2]:
# %% [markdown]
# # --- INITIALIZATION CELL (Run this first!) ---
# This cell downloads and prepares the HAR dataset and the UCF101 video data.

# %% [code]
import os
import time
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import SimpleRNN, LSTM, GRU, Dense, Dropout, Input, TimeDistributed, Bidirectional
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam

# ---------------------------------------------------------
# 1. HAR SENSOR DATA LOADING & PREPROCESSING
# ---------------------------------------------------------
print("Downloading and extracting UCI HAR dataset...")
!wget -q https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip
!unzip -q -o "human+activity+recognition+using+smartphones.zip"
!unzip -q -o "UCI HAR Dataset.zip"

def load_har_data(data_type='train'):
    path = f'UCI HAR Dataset/{data_type}/Inertial Signals/'
    signals = [
        'body_acc_x_', 'body_acc_y_', 'body_acc_z_',
        'body_gyro_x_', 'body_gyro_y_', 'body_gyro_z_',
        'total_acc_x_', 'total_acc_y_', 'total_acc_z_'
    ]
    loaded_signals = []
    for signal in signals:
        filename = f'{path}{signal}{data_type}.txt'
        data = pd.read_csv(filename, delim_whitespace=True, header=None).values
        loaded_signals.append(data)

    X = np.dstack(loaded_signals)
    y = pd.read_csv(f'UCI HAR Dataset/{data_type}/y_{data_type}.txt', delim_whitespace=True, header=None).values
    return X, y

X_train_raw, y_train_raw = load_har_data('train')
X_test_raw, y_test_raw = load_har_data('test')

X_full = np.concatenate((X_train_raw, X_test_raw), axis=0)
y_full = np.concatenate((y_train_raw, y_test_raw), axis=0)

SUBSET_SIZE = 3000
X_sub = X_full[:SUBSET_SIZE]
y_sub = y_full[:SUBSET_SIZE].flatten() - 1

X_train, X_temp, y_train, y_temp = train_test_split(X_sub, y_sub, test_size=0.3, random_state=42, stratify=y_sub)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

scaler = StandardScaler()
X_train_flat = X_train.reshape(-1, X_train.shape[-1])
X_train_scaled = scaler.fit_transform(X_train_flat).reshape(X_train.shape)
X_val_scaled = scaler.transform(X_val.reshape(-1, X_val.shape[-1])).reshape(X_val.shape)
X_test_scaled = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

print(f"HAR Data Loaded. X_train_scaled shape: {X_train_scaled.shape}")

# ---------------------------------------------------------
# 2. UCF101 VIDEO DATA LOADING & PREPROCESSING
# ---------------------------------------------------------
print("\nDownloading and extracting UCF101 Dataset...")
!wget -q --no-check-certificate https://www.crcv.ucf.edu/data/UCF101/UCF101.rar
!unrar x -inul UCF101.rar

classes = ['Basketball', 'Biking', 'Walking']
videos_per_class = 5
num_frames = 10
img_size = (224, 224)

X_videos = []
y_videos = []

print("Extracting frames from videos...")
for label, cls in enumerate(classes):
    video_paths = glob.glob(f'UCF-101/{cls}/*.avi')[:videos_per_class]
    for path in video_paths:
        frames = []
        cap = cv2.VideoCapture(path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames > 0:
            step = max(total_frames // num_frames, 1)
            frame_indices = [i * step for i in range(num_frames)]

            for idx in frame_indices:
                if len(frames) >= num_frames: break
                cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
                ret, frame = cap.read()
                if ret:
                    frame = cv2.resize(frame, img_size)
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frame = frame / 255.0
                    frames.append(frame)
        cap.release()

        while len(frames) < num_frames:
            frames.append(np.zeros((img_size[0], img_size[1], 3)))

        X_videos.append(np.array(frames))
        y_videos.append(label)

videos = np.array(X_videos, dtype='float32')
video_labels = np.array(y_videos)
print(f"Video Data Loaded. videos shape: {videos.shape}")


/tmp/ipykernel_487/2312791382.py:41: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data = pd.read_csv(filename, delim_whitespace=True, header=None).values
/tmp/ipykernel_487/2312791382.py:41: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data = pd.read_csv(filename, delim_whitespace=True, header=None).values
/tmp/ipykernel_487/2312791382.py:41: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data = pd.read_csv(filename, delim_whitespace=True, header=None).values
/tmp/ipykernel_487/2312791382.py:41: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data = pd.read_csv(filename, delim_whitespace=True, header=N

HAR Data Loaded. X_train_scaled shape: (2100, 128, 9)

Extracting frames from videos...
Video Data Loaded. videos shape: (10, 10, 224, 224, 3)


In [3]:
# %% [markdown]
# ## 1 & 2. Effect of Recurrent Units (16, 32, 64) on LSTM vs GRU

# %% [code]
results_ex1_2 = []
units_list = [16, 32, 64]
models_to_test = ['LSTM', 'GRU']

for units in units_list:
    for model_type in models_to_test:
        print(f"\n--- Training {model_type} with {units} units ---")
        model = Sequential()
        model.add(Input(shape=(128, 9)))

        if model_type == 'LSTM':
            model.add(LSTM(units))
        else:
            model.add(GRU(units))

        model.add(Dropout(0.2))
        model.add(Dense(16, activation='relu'))
        model.add(Dense(6, activation='softmax'))

        model.compile(optimizer=Adam(learning_rate=1e-3),
                      loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        start_time = time.time()
        # 15 epochs for quick comparison
        model.fit(X_train_scaled, y_train, validation_data=(X_val_scaled, y_val),
                  epochs=15, batch_size=32, verbose=0)
        training_time = time.time() - start_time

        y_pred_prob = model.predict(X_test_scaled, verbose=0)
        y_pred = np.argmax(y_pred_prob, axis=1)

        acc = accuracy_score(y_test, y_pred)
        _, _, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)

        results_ex1_2.append({
            'Model': f"{model_type}-{units}",
            'Accuracy': acc,
            'Macro F1': f1,
            'Params': model.count_params(),
            'Time (s)': training_time
        })

df_ex1_2 = pd.DataFrame(results_ex1_2)
print("\nResults for varying recurrent units:")
display(df_ex1_2)

# %% [markdown]
# ## 3. Add a second recurrent layer

# %% [code]
print("\n--- Training 2-Layer LSTM ---")
model_2layer = Sequential([
    Input(shape=(128, 9)),
    LSTM(32, return_sequences=True), # First layer must return sequences
    LSTM(32),                        # Second layer
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(6, activation='softmax')
])
model_2layer.compile(optimizer=Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

start_time = time.time()
model_2layer.fit(X_train_scaled, y_train, validation_data=(X_val_scaled, y_val), epochs=15, batch_size=32, verbose=1)
print(f"2-Layer LSTM Time: {time.time() - start_time:.2f}s")

y_pred_2layer = np.argmax(model_2layer.predict(X_test_scaled, verbose=0), axis=1)
f1_2layer = precision_recall_fscore_support(y_test, y_pred_2layer, average='macro', zero_division=0)[2]
print(f"2-Layer LSTM Macro F1: {f1_2layer:.4f}")

# %% [markdown]
# ## 4. Compare a Bidirectional LSTM with Unidirectional LSTM

# %% [code]
print("\n--- Training Bidirectional LSTM ---")
model_bidi = Sequential([
    Input(shape=(128, 9)),
    Bidirectional(LSTM(32)),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(6, activation='softmax')
])
model_bidi.compile(optimizer=Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

start_time = time.time()
model_bidi.fit(X_train_scaled, y_train, validation_data=(X_val_scaled, y_val), epochs=15, batch_size=32, verbose=1)
print(f"Bidirectional LSTM Time: {time.time() - start_time:.2f}s")

y_pred_bidi = np.argmax(model_bidi.predict(X_test_scaled, verbose=0), axis=1)
f1_bidi = precision_recall_fscore_support(y_test, y_pred_bidi, average='macro', zero_division=0)[2]
print(f"Bidirectional LSTM Macro F1: {f1_bidi:.4f}  (Params: {model_bidi.count_params()})")


# %% [markdown]
# ## 5. Sequence Length vs Computational Cost

# %% [code]
seq_lengths = [32, 64, 128]
cost_results = []

for seq_len in seq_lengths:
    X_tr = X_train_scaled[:, :seq_len, :]

    model = Sequential([
        Input(shape=(seq_len, 9)),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(6, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    start_time = time.time()
    # Run for a fixed 5 epochs just to measure speed differences
    model.fit(X_tr, y_train, epochs=5, batch_size=32, verbose=0)
    time_taken = time.time() - start_time

    cost_results.append({'Seq Length': seq_len, 'Time for 5 epochs (s)': time_taken})

df_cost = pd.DataFrame(cost_results)
print("\nComputational Cost by Sequence Length:")
display(df_cost)


# %% [markdown]
# ## 6. Video task: Compare LSTM and GRU (Identical CNN Features)

# %% [code]
print("\n--- Video Task: LSTM vs GRU (Using pre-extracted or frozen CNN) ---")

# CNN-LSTM
video_lstm = Sequential([
    Input(shape=(10, 224, 224, 3)),
    TimeDistributed(MobileNetV2(weights='imagenet', include_top=False, pooling='avg')),
    LSTM(32),
    Dense(5, activation='softmax')
])
video_lstm.layers[0].trainable = False  # Freeze MobileNetV2
video_lstm.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

start_time = time.time()
video_lstm.fit(videos, video_labels, epochs=5, verbose=0)
lstm_time = time.time() - start_time
lstm_acc = accuracy_score(video_labels, np.argmax(video_lstm.predict(videos, verbose=0), axis=1))

# CNN-GRU
video_gru = Sequential([
    Input(shape=(10, 224, 224, 3)),
    TimeDistributed(MobileNetV2(weights='imagenet', include_top=False, pooling='avg')),
    GRU(32),
    Dense(5, activation='softmax')
])
video_gru.layers[0].trainable = False  # Freeze MobileNetV2
video_gru.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

start_time = time.time()
video_gru.fit(videos, video_labels, epochs=5, verbose=0)
gru_time = time.time() - start_time
gru_acc = accuracy_score(video_labels, np.argmax(video_gru.predict(videos, verbose=0), axis=1))

print(f"CNN-LSTM -> Accuracy: {lstm_acc*100:.2f}%, Training Time: {lstm_time:.2f}s")
print(f"CNN-GRU  -> Accuracy: {gru_acc*100:.2f}%, Training Time: {gru_time:.2f}s")


# %% [markdown]
# ## 7. Seq2seq task: Different length for Output Sequence
# e.g., Outputting only the first 2 reversed elements instead of all 4

# %% [code]
input_seq_length = 4
output_seq_length = 2  # Different length! (Predicting just the last 2 digits of the sequence reversed)
vocab_size = 10
num_samples = 5000

# Synthetic dataset: [1, 4, 7, 2] -> [2, 7]
X_seq = np.random.randint(1, vocab_size, size=(num_samples, input_seq_length))
y_seq = np.fliplr(X_seq)[:, :output_seq_length]

# Teacher Forcing Inputs
decoder_input = np.zeros((num_samples, output_seq_length))
decoder_input[:, 0] = 0 # START token
decoder_input[:, 1:] = y_seq[:, :-1]

X_seq_oh = tf.keras.utils.to_categorical(X_seq, num_classes=vocab_size)
decoder_input_oh = tf.keras.utils.to_categorical(decoder_input, num_classes=vocab_size)
y_seq_oh = tf.keras.utils.to_categorical(y_seq, num_classes=vocab_size)

# Encoder (Takes length 4)
enc_inputs = Input(shape=(input_seq_length, vocab_size))
enc_lstm = LSTM(64, return_state=True)
enc_outputs, state_h, state_c = enc_lstm(enc_inputs)

# Decoder (Produces length 2)
dec_inputs = Input(shape=(output_seq_length, vocab_size))
dec_lstm = LSTM(64, return_sequences=True, return_state=True)
dec_outputs, _, _ = dec_lstm(dec_inputs, initial_state=[state_h, state_c])
dec_dense = Dense(vocab_size, activation='softmax')
dec_outputs = dec_dense(dec_outputs)

model_diff_len = Model([enc_inputs, dec_inputs], dec_outputs)
model_diff_len.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("\n--- Training Seq2Seq (Input length 4 -> Output length 2) ---")
model_diff_len.fit([X_seq_oh, decoder_input_oh], y_seq_oh, epochs=10, batch_size=64, validation_split=0.2, verbose=1)

y_pred = np.argmax(model_diff_len.predict([X_seq_oh, decoder_input_oh], verbose=0), axis=-1)
print(f"\nSample Input (len {input_seq_length}): {X_seq[0]}")
print(f"Predicted Output (len {output_seq_length}): {y_pred[0]}")
print(f"Actual Expected Output (len {output_seq_length}): {y_seq[0]}")



--- Training LSTM with 16 units ---

--- Training GRU with 16 units ---

--- Training LSTM with 32 units ---

--- Training GRU with 32 units ---

--- Training LSTM with 64 units ---

--- Training GRU with 64 units ---

Results for varying recurrent units:


,Model,Accuracy,Macro F1,Params,Time (s)
0,LSTM-16,0.922222,0.921181,2038,16.849674
1,GRU-16,0.955556,0.956188,1670,13.404428
2,LSTM-32,0.953333,0.953716,6006,14.265176
3,GRU-32,0.960000,0.960359,4758,12.524441
4,LSTM-64,0.951111,0.950775,20086,13.437829
5,GRU-64,0.962222,0.962938,15542,13.201018



--- Training 2-Layer LSTM ---
Epoch 1/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.4110 - loss: 1.3538 - val_accuracy: 0.7067 - val_loss: 0.8635
Epoch 2/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.8476 - loss: 0.5459 - val_accuracy: 0.9244 - val_loss: 0.3041
Epoch 3/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.8871 - loss: 0.3656 - val_accuracy: 0.9378 - val_loss: 0.2455
Epoch 4/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9324 - loss: 0.2413 - val_accuracy: 0.9422 - val_loss: 0.1643
Epoch 5/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.9524 - loss: 0.1573 - val_accuracy: 0.9444 - val_loss: 0.1675
Epoch 6/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9595 - loss: 0.1274 - val_accuracy: 0.9467 - val_loss: 0.1185
Epoch 7/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9590 - loss: 0.1258 - val_accuracy: 0.9578 - val_loss: 0.0995
Epoch 8/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8500 - loss: 0.

,Seq Length,Time for 5 epochs (s)
0,32,4.577253
1,64,3.629863
2,128,4.121922



--- Video Task: LSTM vs GRU (Using pre-extracted or frozen CNN) ---


/tmp/ipykernel_487/451304103.py:137: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  TimeDistributed(MobileNetV2(weights='imagenet', include_top=False, pooling='avg')),


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/tmp/ipykernel_487/451304103.py:152: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  TimeDistributed(MobileNetV2(weights='imagenet', include_top=False, pooling='avg')),


CNN-LSTM -> Accuracy: 100.00%, Training Time: 41.89s
CNN-GRU  -> Accuracy: 100.00%, Training Time: 38.81s

--- Training Seq2Seq (Input length 4 -> Output length 2) ---
Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.3674 - loss: 2.1432 - val_accuracy: 0.4125 - val_loss: 1.8713
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5121 - loss: 1.3844 - val_accuracy: 0.6005 - val_loss: 1.0277
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7009 - loss: 0.8039 - val_accuracy: 0.7845 - val_loss: 0.6480
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8720 - loss: 0.4825 - val_accuracy: 0.9320 - val_loss: 0.3746
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9797 - loss: 0.2667 - val_accuracy: 0.9970 - val_loss: 0.1968
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9992 - loss: 0.1389 - val_accuracy: 1.0000 - val_loss: 0.1021
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 1.0000 - lo


Sample Input (len 4): [9 7 3 5]
Predicted Output (len 2): [5 3]
Actual Expected Output (len 2): [5 3]
